# SRC LEARNING · giao diện Desktop trên Colab

Mở notebook trên Colab, chạy hai ô theo thứ tự. Ứng dụng Streamlit xuất hiện **ngay trong output notebook**. Dán Google Drive File ID của môn đã được phê duyệt để tải dữ liệu; sau đó dùng Flash Cards, Học cấp tốc và Mock Exam.

**Phiên mới:** Ứng dụng tạo DATA và SQLite tạm trong runtime Colab. Khi runtime bị xóa, dữ liệu và tiến độ cũng có thể mất. Không có đồng bộ với Desktop.

**Colab miễn phí:** Google có thể ngắt runtime chạy web UI; cách mở Streamlit bằng iframe này là thử nghiệm, không bảo đảm hoạt động ổn định hoặc luôn mở được trên Safari/iPad. Không dùng tunnel hay máy chủ ngoài Colab. Ảnh/đáp án có thể hiển thị trong output notebook; không chia sẻ notebook kèm output dữ liệu.


In [ ]:
from pathlib import Path
import subprocess, sys, tempfile
REPO_URL = "https://github.com/ScottPhantom/SRC-LEARNING.git"
BRANCH = "Mobile_Web_APP"
SOURCE_DIR = Path(tempfile.mkdtemp(prefix="src-learning-source-")) / "SRC-LEARNING"
subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(SOURCE_DIR)], check=True)
actual_branch = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "branch", "--show-current"], text=True).strip()
assert actual_branch == BRANCH, f"Đã tải nhánh khác: {actual_branch}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(SOURCE_DIR / "requirements-mobile.txt")], check=True)
print("Đã tải nhánh", actual_branch, "commit", subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import os, subprocess, sys, tempfile, time, urllib.request
from google.colab import output

if "streamlit_process" in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
SESSION_ROOT = tempfile.mkdtemp(prefix="src-learning-session-")
streamlit_process = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(SOURCE_DIR / "colab_mobile" / "streamlit_app.py"), "--server.headless", "true", "--server.address", "127.0.0.1", "--server.port", "8501"], cwd=SOURCE_DIR, env={**os.environ, "SRC_LEARNING_SESSION_ROOT": SESSION_ROOT}, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(60):
    if streamlit_process.poll() is not None:
        raise RuntimeError("Streamlit đã dừng trước khi sẵn sàng")
    try:
        urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2).read()
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Streamlit không sẵn sàng sau 60 giây")
output.serve_kernel_port_as_iframe(8501, height=960)
